In [14]:
import pickle
import pandas as pd
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential

# Load and preprocess data
data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

onehot_encoder_geo = OneHotEncoder(handle_unknown="ignore")
geo_encoded = onehot_encoder_geo.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(["Geography"]),
)

data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)

X = data.drop("Exited", axis=1)
y = data["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Cleaned model creation function using Input(shape=...)
def create_model(neurons=32, layers=1):
  model = Sequential()
  # FIX: Use Input layer explicitly to remove Keras 3 UserWarning
  model.add(Input(shape=(X_train.shape[1],)))

  for _ in range(layers):
    model.add(Dense(neurons, activation="relu"))

  model.add(Dense(1, activation="sigmoid"))
  model.compile(
      optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
  )
  return model


model = KerasClassifier(
    model=create_model, epochs=20, batch_size=32, verbose=0
)

# Reduced parameter search space for faster verification
param_grid = {
    "model__neurons": [16, 32],
    "model__layers": [1, 2],
    "epochs": [20, 30],
}

grid = GridSearchCV(
    estimator=model, param_grid=param_grid, n_jobs=1, cv=3, verbose=1
)
grid_result = grid.fit(X_train, y_train)

print(
    "Best score: %f using %s"
    % (grid_result.best_score_, grid_result.best_params_)
)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best score: 0.857875 using {'epochs': 30, 'model__layers': 1, 'model__neurons': 16}
